# Login + OTP + SQLite + Joke → QRCode (Colab)


**Instructions**  
1. Run each cell from top to bottom.  
2. Register a user (you may change the sample username/password/email).  
3. Scan the OTP QR with Google Authenticator.  
4. Use **username + password + OTP** to log in.  
5. After successful login, run the **Get Joke** and **Joke → QR** cells (they are **locked** unless you're logged in).


In [ ]:

# ✅ If running in Google Colab, install required packages
# (Skip locally if already installed)
!pip -q install pyotp qrcode[pil]


In [ ]:

import sqlite3

# Connect to SQLite DB (file persists in the session / runtime)
conn = sqlite3.connect("users.db")
cursor = conn.cursor()

cursor.execute('''
CREATE TABLE IF NOT EXISTS users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT UNIQUE,
    password TEXT,
    email TEXT,
    secret TEXT
)
''')
conn.commit()
print("✅ SQLite table ready.")


In [ ]:

import pyotp

def register_user(username: str, password: str, email: str) -> str:
    """Registers/overwrites a user and returns a new OTP secret."""
    secret = pyotp.random_base32()
    cursor.execute(
        "INSERT OR REPLACE INTO users (username, password, email, secret) VALUES (?, ?, ?, ?)",
        (username, password, email, secret)
    )
    conn.commit()
    print(f"✅ Registered '{username}' (email: {email}) with secret: {secret}")
    return secret

# ▶️ CHANGE THESE or keep defaults for a demo
USERNAME = "testuser"
PASSWORD = "testpass"
EMAIL = "test@email.com"

secret = register_user(USERNAME, PASSWORD, EMAIL)


In [ ]:

import qrcode
from IPython.display import Image, display

def get_secret(username: str) -> str:
    cursor.execute("SELECT secret FROM users WHERE username = ?", (username,))
    row = cursor.fetchone()
    if not row:
        raise ValueError("User not found. Register first.")
    return row[0]

def generate_otp_qr(username: str, secret: str, issuer: str = "MyApp"):
    totp = pyotp.TOTP(secret)
    uri = totp.provisioning_uri(name=username, issuer_name=issuer)
    img = qrcode.make(uri)
    img.save("otp_qr.png")
    display(Image(filename="otp_qr.png"))
    print("📱 Scan this QR in Google Authenticator.")

secret = get_secret(USERNAME)
generate_otp_qr(USERNAME, secret)


In [ ]:

AUTHENTICATED = False
CURRENT_USER = None

def login(username: str, password: str, otp_code: str) -> bool:
    cursor.execute("SELECT password, secret FROM users WHERE username=?", (username,))
    row = cursor.fetchone()
    if not row:
        print("❌ User not found")
        return False
    stored_password, secret = row
    if password != stored_password:
        print("❌ Invalid password")
        return False
    totp = pyotp.TOTP(secret)
    if not totp.verify(otp_code):
        print("❌ Invalid OTP")
        return False
    print("✅ Login successful!")
    return True

# ▶️ Enter your credentials and the current OTP from Google Authenticator
input_username = input("Username: ").strip()
input_password = input("Password: ").strip()
input_otp = input("OTP (from Google Authenticator): ").strip()

if login(input_username, input_password, input_otp):
    AUTHENTICATED = True
    CURRENT_USER = input_username
else:
    AUTHENTICATED = False
    CURRENT_USER = None

print("AUTHENTICATED =", AUTHENTICATED)


In [ ]:

import requests

def get_joke() -> str:
    url = "https://official-joke-api.appspot.com/random_joke"
    r = requests.get(url, timeout=15)
    r.raise_for_status()
    data = r.json()
    joke = f"{data['setup']} - {data['punchline']}"
    print("😂 Joke:", joke)
    return joke

if not AUTHENTICATED:
    raise PermissionError("🔒 Please log in first (run the previous cell with valid username/password/OTP).")

joke_text = get_joke()


In [ ]:

def generate_joke_qr(text: str, filename: str = "joke_qr.png"):
    img = qrcode.make(text)
    img.save(filename)
    display(Image(filename=filename))
    print(f"✅ QR Code generated and saved as '{filename}'.")

if not AUTHENTICATED:
    raise PermissionError("🔒 Please log in first (run the login cell).")

generate_joke_qr(joke_text)
